# 02b — Train SAE on cached activations
### Driver is thin, heavy lifting is done in the mid.sae.train_sae
Owner: David Teklea

In [1]:
import sys
import torch
sys.path.insert(0, "../src")

from mid.sae.activations import read_activations
from mid.sae.load import load_sae
from mid.sae.train_sae import train_sae
from mid.config import load_configs, load_sae_config


In [2]:
#Model type: String: [decoder, encoder, or encoderdecoder]  Default: decoder
model_type = "decoder"
#Model size: String: [small, medium, large]  Default: small
model_size = "small"

#Use split: Bool: [True, False]  Default: True
#Whether to split data into train and validation for the SAE or to use all data.
use_split = True # True -> train.pt; False -> all.pt

model_config_path = f"../configs/model_{model_type}_{model_size}.yaml"
sae_config_path = f"../configs/sae_{model_type}_{model_size}.yaml"
checkpoint_path = f"../checkpoints/{model_type}_{model_size}_best_model.pt"
activations_pt_path = (
    f"../data/sae_data/{model_type}_{model_size}/train.pt" if use_split
    else f"../data/sae_data/{model_type}_{model_size}/all.pt"
)

model_cfg, _ = load_configs(model_config_path)
sae_cfg = load_sae_config(sae_config_path)
print(model_cfg)
print(sae_cfg)


ModelConfig(d_model=128, d_head=32, n_heads=4, n_layers=4, d_mlp=512, d_vocab=3000, n_ctx=256, act_fn='gelu', normalization_type='LN', tokenizer_name=None, seed=16)
SAEConfig(lr=0.0003, batch_size=4096, training_tokens=10000000, context_len=256, l1_coeff=2.0, l1_warmup_steps=500, lr_warmup_steps=100, apply_bias_decay_to_input=True, normalize_activations='expected_average_only_in', dim_input=128, dim_sae=1024, hook_type='stream', layer=3, num_batches_in_buffer=32, store_batch_size_prompts=16, seed=32)


In [3]:
#Collect input data

#Specify the layer here, or you can make this a for loop across all layers
layer = 0

if use_split:
    #Training
    get_path = f"../data/sae_data/{model_type}_{model_size}/train.pt"
    train_activations, train_metadata, train_num_tokens, train_num_activations = read_activations(get_path, layer)
    #Validation
    get_path = f"../data/sae_data/{model_type}_{model_size}/val.pt"
    train_activations, train_metadata, train_num_tokens, train_num_activations = read_activations(get_path, layer)

else:
    #All, no validation, saved as training to avoid having to have another variable name.
    get_path = f"../data/sae_data/{model_type}_{model_size}/all.pt"
    train_activations, train_metadata, train_num_tokens, train_num_activations = read_activations(get_path, layer)

In [4]:
# Train a single SAE. Output goes to `../data/sae_out/{model_type}_{model_size}/L{layer}_{hook_type}/`.

In [5]:
out_dir = (
    f"../data/sae_out/{model_type}_{model_size}/L{sae_cfg.layer}_{sae_cfg.hook_type}"
)

sae = train_sae(
    sae_cfg=sae_cfg,
    model_cfg=model_cfg,
    checkpoint_path=checkpoint_path,
    activations_pt_path=activations_pt_path,
    out_dir=out_dir,
)

Training SAE on device: cuda


Saving the dataset (0/2 shards):   0%|          | 0/5833 [00:00<?, ? examples/s]

Repacked cache from 5833 seqs x 256 tokens/seq x 128 d_in to HF Dataset at ..\data\sae_out\decoder_small\L3_stream\saelens_cache.at hook 'blocks.3.hook_resid_post' -> ..\data\sae_out\decoder_small\L3_stream\saelens_cache


You just passed in a dataset which will override the one specified in your configuration: shakespeare-cached. As a consequence this run will not be reproducible via configuration alone.
You just passed in a model which will override the one specified in your configuration: shakespeare-decoder-small. As a consequence this run will not be reproducible via configuration alone.


Moving model to device:  cuda
Model parameters: 1,597,112
Moving model to device:  cuda


C:\Users\mjcoc\miniconda3\envs\shakespeare_mech_interp\Lib\site-packages\sae_lens\training\activations_store.py:261: UserWarning: The training dataset contains fewer samples (5833) than the number of samples required by your training configuration (10000000). This will result in multiple training epochs and some samples being used more than once.
  warnings.warn(


Training SAE:   0%|          | 0/10000000 [00:00<?, ?it/s]

Estimating norm scaling factor:   0%|          | 0/1000 [00:00<?, ?it/s]

C:\Users\mjcoc\miniconda3\envs\shakespeare_mech_interp\Lib\site-packages\sae_lens\training\activations_store.py:709: UserWarning: All samples in the training dataset have been exhausted, beginning new epoch.
  warnings.warn(


SAE training complete. Saved to ..\data\sae_out\decoder_small\L3_stream


In [6]:
# Smoke test: reload the SAE from disk and check reconstruction + sparsity on
# a slice of the cached train activations. Feature density (L0) should be a small fraction
# of d_sae; variance-explained should be close to 1.

device = "cuda" if torch.cuda.is_available() else "cpu"
sae = load_sae(out_dir, device=device)

activations, _, _, _ = read_activations(activations_pt_path, layer_num=sae_cfg.layer)
x = activations[:4096].to(device, dtype=torch.float32)

with torch.no_grad():
    feats = sae.encode(x)
    x_hat = sae.decode(feats)

l0 = (feats > 0).float().sum(dim=-1).mean().item()
mse = torch.nn.functional.mse_loss(x_hat, x).item()
var_explained = 1 - ((x - x_hat).var() / x.var()).item()
print(f"L0 (mean active features per token): {l0:.1f} / {sae_cfg.dim_sae}")
print(f"Reconstruction MSE: {mse:.4f}")
print(f"Variance explained: {var_explained:.3f}")

C:\Users\mjcoc\miniconda3\envs\shakespeare_mech_interp\Lib\site-packages\sae_lens\saes\sae.py:251: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


L0 (mean active features per token): 29.3 / 1024
Reconstruction MSE: 0.3123
Variance explained: 0.790
